# Pre-Processing

In [6]:
# Pre-Processing — Unified Model Pipeline (with robust NaN/Inf handling)
# Goal -> Transform wrangled table into ONE model-ready bundle for a unified neural network
# - Keep bucket-aware capping/imputation for stability within each regime
# - Combine both buckets before split/scaler/cat-maps so transforms fit on the combined TRAIN
# - Include `has_claims_features` (bucket flag) as a numeric feature
#
# Outputs (saved under curated/training/):
#   - processed_unified.npz with X_train/valid/test, Z_train/valid/test, y_*, and B_* (bucket masks)
#   - cat_index_maps_unified.json for embedding models (per-categorical index maps)
#   - feature_manifest_unified.json with feature names, standardization stats, cardinalities, pos_weight, etc.
#
# Notes:
# - Assumes wrangling notebook already wrote parquet: curated/training/providers_nn_asof_YYYY-MM-DD.parquet
# - Optionally reuses EDA artifacts if present: caps_YYYY-MM-DD.json and impute_map_YYYY-MM-DD.json
# - Identifier-like columns (NPI, etc.) are *not* used as features and are carried only for joins/QA

from __future__ import annotations
import json
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# -------------------------
# Config
# -------------------------
AS_OF_STR = "2023-12-31"  # keep in sync with wrangling/EDA
CURATED_DIR = Path("curated/training")
PARQUET_PATH = CURATED_DIR / f"providers_nn_asof_{AS_OF_STR}.parquet"
CAPS_PATH    = CURATED_DIR / f"caps_{AS_OF_STR}.json"
IMPUTE_PATH  = CURATED_DIR / f"impute_map_{AS_OF_STR}.json"
POOLED_PATH  = CURATED_DIR / f"pooled_levels_{AS_OF_STR}.json"  # optional

RANDOM_STATE = 42
np.random.seed(42)
TEST_SIZE    = 0.15
VALID_SIZE   = 0.15  # of the *remaining* after test split

ID_LIKE_COLS = ["npi", "tin", "ein", "org_id"]
TARGET_COL   = "label_fraud"  # from EDA mapping
BUCKET_COL   = "has_claims_features"

# Prefer pooled categoricals if EDA created them, else fall back to base
BASE_CAT_COLS   = ["state", "entity_type", "taxonomy"]
POOLED_SUFFIX   = "_rare_pooled"

# Numeric columns we expect (subset will be auto-detected if missing)
NUM_CANDIDATES = [
    "services_total","beneficiaries_total","beneficiary_days_total",
    "hcpcs_unique_count","drug_services_rate","missing_zip_rate",
    "avg_charge_w","avg_allowed_w","avg_payment_w","charge_std_w",
    "charge_to_allowed","payment_to_allowed","services_per_beneficiary",
    "beneficiary_days_per_beneficiary","days_since_enrollment",
    # EDA-derived
    "claims_feat_coverage","claims_volume",
]

# -------------------------
# Utilities
# -------------------------

def _exists(p: Path) -> bool:
    try:
        return p.exists()
    except Exception:
        return False

@dataclass
class FeatureSpec:
    numeric: List[str]
    categorical: List[str]


def load_artifacts() -> Tuple[Dict, Dict, Dict]:
    """Load optional EDA artifacts (caps, impute_map, pooled levels)."""
    caps = json.loads(Path(CAPS_PATH).read_text()) if _exists(CAPS_PATH) else {}
    impute_map = json.loads(Path(IMPUTE_PATH).read_text()) if _exists(IMPUTE_PATH) else {}
    pooled = json.loads(Path(POOLED_PATH).read_text()) if _exists(POOLED_PATH) else {}
    return caps, impute_map, pooled


def choose_categoricals(df: pd.DataFrame) -> List[str]:
    # Prefer pooled versions if they exist
    cats = []
    for base in BASE_CAT_COLS:
        pooled = f"{base}{POOLED_SUFFIX}"
        if pooled in df.columns:
            cats.append(pooled)
        elif base in df.columns:
            cats.append(base)
    return cats


def detect_numerics(df: pd.DataFrame) -> List[str]:
    # Take intersection of NUM_CANDIDATES and actual numeric columns
    present = [c for c in NUM_CANDIDATES if c in df.columns]
    # Add any other numeric columns that are not identifiers/target/bucket
    extra = [c for c in df.select_dtypes(include=[np.number]).columns
             if c not in present + [TARGET_COL, BUCKET_COL] + ID_LIKE_COLS]
    # Dedup preserve order
    seen = set()
    out = []
    for c in present + extra:
        if c not in seen:
            out.append(c); seen.add(c)
    return out


def apply_caps(df: pd.DataFrame, caps: Dict, bucket: int) -> pd.DataFrame:
    if not caps:
        return df
    df = df.copy()
    for col, by_bucket in caps.items():
        if col in df.columns and str(bucket) in by_bucket:
            upper = by_bucket[str(bucket)]
            df.loc[df[BUCKET_COL] == bucket, col] = (
                df.loc[df[BUCKET_COL] == bucket, col].clip(upper=upper)
            )
    return df


def apply_impute(df: pd.DataFrame, impute_map: Dict, bucket: int) -> pd.DataFrame:
    """Bucket-aware imputation with robust guards (NaN medians -> 0.0)."""
    df = df.copy()
    idx = df[BUCKET_COL] == bucket

    if impute_map:
        # Numeric via provided map
        for c, by_bucket in impute_map.items():
            if c in df.columns and str(bucket) in by_bucket:
                val = by_bucket[str(bucket)]
                if val is None or (isinstance(val, float) and np.isnan(val)):
                    val = 0.0
                df.loc[idx, c] = df.loc[idx, c].fillna(val)
    else:
        # Fallback: conservative median per bucket, with NaN-guard
        num_cols = df.select_dtypes(include=[np.number]).columns
        for c in num_cols:
            if c in [TARGET_COL] + ID_LIKE_COLS:
                continue
            med = df.loc[idx, c].median()
            if pd.isna(med):
                med = 0.0
            df.loc[idx, c] = df.loc[idx, c].fillna(med)

    # Categoricals
    for c in df.select_dtypes(exclude=[np.number]).columns:
        df[c] = df[c].fillna("MISSING")

    return df


def build_cat_index(df: pd.DataFrame, cat_cols: List[str]) -> Dict[str, Dict[str, int]]:
    """Create integer index maps for each categorical, reserving 0 for 'OOV'/'MISSING'."""
    maps: Dict[str, Dict[str, int]] = {}
    for c in cat_cols:
        levels = df[c].astype(str).fillna("MISSING").unique().tolist()
        levels = sorted(set(levels))
        idx = {"<OOV>": 0}
        for i, lvl in enumerate(levels, start=1):
            idx[str(lvl)] = i
        maps[c] = idx
    return maps


def encode_categoricals_to_indices(df: pd.DataFrame, cat_maps: Dict[str, Dict[str, int]]) -> np.ndarray:
    mats = []
    for c, mp in cat_maps.items():
        col = df[c].astype(str).fillna("MISSING").map(lambda v: mp.get(str(v), 0)).to_numpy(dtype=np.int64)
        mats.append(col.reshape(-1, 1))
    return np.hstack(mats) if mats else np.empty((len(df), 0), dtype=np.int64)


def standardize_numeric(df: pd.DataFrame, num_cols: List[str], fit_on: Optional[pd.DataFrame]=None) -> Tuple[np.ndarray, Dict[str, Tuple[float,float]]]:
    """Return standardized numeric matrix and stats {col: (mean, std)} with hard NaN/Inf guards."""
    stats = {}
    if not num_cols:
        return np.empty((len(df), 0), dtype=np.float32), stats

    ref = fit_on if fit_on is not None else df
    ref = ref.copy()
    # Guard: ensure no NaNs in ref columns used for stats
    for c in num_cols:
        if ref[c].isna().all():
            ref[c] = 0.0
        else:
            ref[c] = ref[c].fillna(0.0)

    means = ref[num_cols].mean()
    stds  = ref[num_cols].std().replace(0, 1.0).fillna(1.0)

    X = (df[num_cols].fillna(0.0) - means) / stds
    for c in num_cols:
        m = float(means[c]) if not pd.isna(means[c]) else 0.0
        s = float(stds[c])  if not pd.isna(stds[c])  else 1.0
        stats[c] = (m, s)
    return X.to_numpy(dtype=np.float32), stats


def make_feature_spec(df: pd.DataFrame) -> FeatureSpec:
    cat = choose_categoricals(df)
    num = detect_numerics(df)
    # Drop any columns that clash with target/bucket or are id-like
    num  = [c for c in num if c not in [TARGET_COL, BUCKET_COL] + ID_LIKE_COLS]
    cat  = [c for c in cat if c not in [TARGET_COL, BUCKET_COL] + ID_LIKE_COLS]
    return FeatureSpec(numeric=num, categorical=cat)


def _safe_train_valid_test_split(frame, y, test_size, valid_size, rs):
    try:
        df_tmp, df_test, y_tmp, y_test = train_test_split(
            frame, y, test_size=test_size, random_state=rs, stratify=y
        )
        valid_ratio = valid_size / (1.0 - test_size)
        df_train, df_valid, y_train, y_valid = train_test_split(
            df_tmp, y_tmp, test_size=valid_ratio, random_state=rs, stratify=y_tmp
        )
    except ValueError:
        # fallback if a class is missing in a split
        df_tmp, df_test, y_tmp, y_test = train_test_split(
            frame, y, test_size=test_size, random_state=rs, stratify=None
        )
        valid_ratio = valid_size / (1.0 - test_size)
        df_train, df_valid, y_train, y_valid = train_test_split(
            df_tmp, y_tmp, test_size=valid_ratio, random_state=rs, stratify=None
        )
    return df_train, df_valid, df_test, y_train, y_valid, y_test

# -------------------------
# Unified preprocessing entry
# -------------------------

def preprocess_unified(
    df: pd.DataFrame,
    test_size: float = TEST_SIZE,
    valid_size: float = VALID_SIZE,
    random_state: int = RANDOM_STATE,
    use_caps: bool = True,
    use_impute_map: bool = True,
):
    """
    Unified pipeline:
      1) Apply caps/impute per bucket (stability within each regime).
      2) Concatenate buckets back together.
      3) Split train/valid/test on the combined set (stratified).
      4) Fit cat maps + numeric scaler on combined TRAIN.
      5) Emit arrays + per-bucket masks for metric slicing.
    """
    caps, impute_map, _ = load_artifacts()

    # 1) bucket-aware cleaning
    df0 = df[df[BUCKET_COL] == 0].copy()
    df1 = df[df[BUCKET_COL] == 1].copy()
    if use_caps:
        df0 = apply_caps(df0, caps, 0)
        df1 = apply_caps(df1, caps, 1)
    if use_impute_map:
        df0 = apply_impute(df0, impute_map, 0)
        df1 = apply_impute(df1, impute_map, 1)
    else:
        df0 = apply_impute(df0, {}, 0)
        df1 = apply_impute(df1, {}, 1)

    # 2) recombine, scrub NaN/Inf aggressively
    dfx = pd.concat([df0, df1], ignore_index=True)
    dfx.replace([np.inf, -np.inf], np.nan, inplace=True)

    # Build a probe spec to identify numeric columns to scrub
    spec_probe = make_feature_spec(dfx)
    num_cols_all = list(set(spec_probe.numeric + [BUCKET_COL]))
    for c in [c for c in num_cols_all if c in dfx.columns]:
        if dfx[c].isna().all():
            dfx[c] = 0.0
        else:
            dfx[c] = dfx[c].fillna(0.0)

    # guards
    assert TARGET_COL in dfx.columns, f"Missing target column: {TARGET_COL}"
    assert BUCKET_COL in dfx.columns, f"Missing bucket column: {BUCKET_COL}"
    dfx[TARGET_COL] = dfx[TARGET_COL].astype(int)
    dfx[BUCKET_COL] = dfx[BUCKET_COL].astype(int)
    y = dfx[TARGET_COL].to_numpy()

    # 3) feature spec (include bucket as a numeric feature!)
    spec = make_feature_spec(dfx)
    if BUCKET_COL not in spec.numeric:
        spec.numeric = spec.numeric + [BUCKET_COL]

    # split (combined)
    df_train, df_valid, df_test, y_train, y_valid, y_test = _safe_train_valid_test_split(
        dfx, y, test_size, valid_size, random_state
    )

    # keep bucket masks for metric slicing later
    B_train = df_train[BUCKET_COL].to_numpy(np.int64)
    B_valid = df_valid[BUCKET_COL].to_numpy(np.int64)
    B_test  = df_test[BUCKET_COL].to_numpy(np.int64)

    # 4) cat maps on combined TRAIN
    cat_maps = build_cat_index(df_train, spec.categorical)

    # 5) numeric standardization on TRAIN (includes bucket flag; fine to standardize)
    Xn_train, num_stats = standardize_numeric(df_train, spec.numeric, fit_on=df_train)
    Xn_valid, _ = standardize_numeric(df_valid, spec.numeric, fit_on=df_train)
    Xn_test,  _ = standardize_numeric(df_test,  spec.numeric, fit_on=df_train)

    # categorical indices
    Z_train = encode_categoricals_to_indices(df_train, cat_maps)
    Z_valid = encode_categoricals_to_indices(df_valid, cat_maps)
    Z_test  = encode_categoricals_to_indices(df_test,  cat_maps)

    # arrays
    X_train, X_valid, X_test = Xn_train, Xn_valid, Xn_test

    # sanity checks
    for name, mat in [("X_train", X_train), ("X_valid", X_valid), ("X_test", X_test)]:
        assert np.isfinite(mat).all(), f"{name} contains NaN/Inf"
    for name, mat in [("Z_train", Z_train), ("Z_valid", Z_valid), ("Z_test", Z_test)]:
        assert mat.dtype == np.int64, f"{name} should be int64"

    # class stats for pos_weight
    pos_ct = int(y_train.sum()); neg_ct = int(len(y_train) - pos_ct)
    pos_rate = float(pos_ct / max(1, pos_ct + neg_ct))
    pos_weight = float(neg_ct / max(1, pos_ct)) if pos_ct > 0 else 1.0

    feature_manifest = {
        "as_of": AS_OF_STR,
        "setup": "unified",
        "target_col": TARGET_COL,
        "bucket_col": BUCKET_COL,
        "id_like_cols": [c for c in ID_LIKE_COLS if c in df.columns],
        "numeric_features": spec.numeric,                     # includes BUCKET_COL at the end
        "categorical_features": spec.categorical,
        "numeric_standardization": num_stats,
        "categorical_index_cardinalities": {c: int(max(mp.values()) + 1 if mp else 1) for c, mp in cat_maps.items()},
        "n_train": int(len(df_train)), "n_valid": int(len(df_valid)), "n_test": int(len(df_test)),
        "train_pos": pos_ct, "train_neg": neg_ct,
        "train_pos_rate": pos_rate, "pos_weight": pos_weight
    }

    return {
        "X_train": X_train, "y_train": y_train, "B_train": B_train,
        "X_valid": X_valid, "y_valid": y_valid, "B_valid": B_valid,
        "X_test":  X_test,  "y_test":  y_test,  "B_test":  B_test,
        "Z_train": Z_train, "Z_valid": Z_valid, "Z_test": Z_test,
        "feature_manifest": feature_manifest,
        "cat_index_maps": cat_maps,
    }


# -------------------------
# Top-level script block
# -------------------------
if __name__ == "__main__":
    print("Loading wrangled parquet:", PARQUET_PATH)
    df = pd.read_parquet(PARQUET_PATH, engine="pyarrow")

    # Harmonize column names to the EDA-friendly names if needed
    rename_map = {
        "state_abbr": "state",
        "entity_type_code": "entity_type",
        "primary_taxonomy": "taxonomy",
        "npi_age_days": "days_since_enrollment",
        "is_active": "npi_active",
        "is_organization_subpart": "org_subpart_flag",
        "is_sole_proprietor": "sole_proprietor_flag",
        "total_services": "services_total",
        "total_beneficiaries": "beneficiaries_total",
        "total_bene_day_services": "beneficiary_days_total",
        "num_unique_procedures": "hcpcs_unique_count",
        "frac_drug_services": "drug_services_rate",
        "frac_missing_zip": "missing_zip_rate",
        "w_avg_submitted_charge": "avg_charge_w",
        "w_avg_allowed": "avg_allowed_w",
        "w_avg_payment": "avg_payment_w",
        "w_stddev_submitted_charge": "charge_std_w",
        "charge_allowed_ratio": "charge_to_allowed",
        "payment_allowed_ratio": "payment_to_allowed",
        "services_per_bene": "services_per_beneficiary",
        "bene_days_per_bene": "beneficiary_days_per_beneficiary",
        "has_puf": "has_claims_features",
        "is_fraud": "label_fraud",
        "is_excluded_asof": "status_excluded_asof",
    }
    common = {k: v for k, v in rename_map.items() if k in df.columns}
    if common:
        df = df.rename(columns=common)

    # Guards
    for c in [TARGET_COL, BUCKET_COL, "npi"]:
        assert c in df.columns, f"Required column missing: {c}"

    # Ensure ints
    df[BUCKET_COL] = df[BUCKET_COL].fillna(0).astype(int)
    df[TARGET_COL] = df[TARGET_COL].fillna(0).astype(int)

    print("\n=== Processing UNIFIED dataset ===")
    pack = preprocess_unified(df)

    # Save arrays
    CURATED_DIR.mkdir(parents=True, exist_ok=True)
    out_npz = CURATED_DIR / "processed_unified.npz"
    np.savez_compressed(
        out_npz,
        X_train=pack["X_train"], y_train=pack["y_train"], B_train=pack["B_train"],
        X_valid=pack["X_valid"], y_valid=pack["y_valid"], B_valid=pack["B_valid"],
        X_test=pack["X_test"],   y_test=pack["y_test"],   B_test=pack["B_test"],
        Z_train=pack["Z_train"], Z_valid=pack["Z_valid"], Z_test=pack["Z_test"],
    )
    print("Saved arrays →", out_npz)

    # Save cat maps & manifest
    idx_path = CURATED_DIR / "cat_index_maps_unified.json"
    with open(idx_path, "w") as f:
        json.dump(pack["cat_index_maps"], f, indent=2)
    man_path = CURATED_DIR / "feature_manifest_unified.json"
    with open(man_path, "w") as f:
        json.dump(pack["feature_manifest"], f, indent=2)
    print("Saved metadata →", idx_path, "&", man_path)

    # Quick smoke
    y = pack["y_train"]; b = pack["B_train"]
    print(f"\nUNIFIED → X_train {pack['X_train'].shape}, Z_train {pack['Z_train'].shape}, "
          f"y+ {y.sum()}/{len(y)} ({y.mean():.3%}); "
          f"bucket1 share in train = {(b==1).mean():.2%}")


Loading wrangled parquet: curated/training/providers_nn_asof_2023-12-31.parquet

=== Processing UNIFIED dataset ===
Saved arrays → curated/training/processed_unified.npz
Saved metadata → curated/training/cat_index_maps_unified.json & curated/training/feature_manifest_unified.json

UNIFIED → X_train (134189, 24), Z_train (134189, 3), y+ 4697/134189 (3.500%); bucket1 share in train = 13.92%
